## 🎯 Learning Objectives
* Understand the critical role of unit testing in developing robust and reliable AI agent tools.
* Learn how to write effective unit tests for agent tools using `pytest`.
* Master techniques for mocking external dependencies (e.g., API calls, LLM interactions) to isolate tool logic during testing.
* Identify common pitfalls and best practices for testing agent tools in an agentic workflow.


## Unit Testing Agent Tools: Ensuring Reliability in Agentic Workflows

In the rapidly evolving landscape of AI agents, reliability is paramount. An agent's ability to perform complex tasks hinges on the correctness and robustness of its individual components, especially its *tools*. Just as a master craftsman relies on perfectly calibrated instruments, an AI agent relies on well-tested tools to achieve its objectives.

### What are Agent Tools and Why Test Them?

Agent tools are essentially specialized functions or microservices that an AI agent (typically powered by a Large Language Model, or LLM) can invoke to interact with the external world. This could involve fetching real-time data, executing code, sending emails, or interacting with databases. They are the agent's 'hands' and 'eyes'.

**Why unit test them?**

Imagine an autonomous vehicle. Each sensor, each braking mechanism, each navigation algorithm must work flawlessly in isolation before they are integrated into the complex system. Similarly, an agent's tools must be rigorously tested in isolation to ensure:

1.  **Correctness**: Does the tool produce the expected output for a given input?
2.  **Robustness**: Does it handle invalid inputs, edge cases, and error conditions gracefully?
3.  **Reliability**: Will it consistently perform its function without unexpected failures?
4.  **Maintainability**: Can we refactor or update the tool's internal logic without breaking its external contract?

Without unit tests, debugging agent failures becomes a nightmare. Was it the LLM's reasoning? Was it a faulty tool? Unit tests help pinpoint issues quickly by isolating the tool's logic.

### The Challenge: External Dependencies and Non-Determinism

Testing agent tools presents unique challenges compared to traditional software:

*   **External APIs**: Many tools interact with external services (e.g., weather APIs, stock market data, internal company APIs). Directly calling these in tests is slow, costly, and can lead to non-deterministic results if the external service changes.
*   **LLM Interaction (Indirect)**: While we're unit testing the *tool* itself, not the LLM's ability to *use* the tool, the tool's design might be influenced by how an LLM expects to interact with it (e.g., input schema, output format). However, for unit tests, we focus purely on the tool's internal logic.
*   **Statefulness**: Some tools might maintain internal state, which needs careful handling during testing to ensure isolation between test cases.

### The Solution: Isolation and Mocking with `pytest`

We tackle these challenges by adhering to the principle of **isolation**. Unit tests should test a single unit of code (the tool's core logic) in isolation from its dependencies. This is achieved through **mocking**.

**Mocking** involves replacing real external dependencies (like an API call) with controlled, simulated objects that return predefined responses. This allows us to:

*   **Control outcomes**: Simulate success, failure, specific data responses.
*   **Speed up tests**: Avoid slow network calls.
*   **Ensure determinism**: Tests always produce the same result.
*   **Reduce costs**: Avoid hitting rate limits or incurring charges for external API usage.

We'll use `pytest`, a powerful and popular Python testing framework, along with `unittest.mock` (built into Python) to demonstrate these concepts. By 2026, `pytest` remains the de-facto standard for Python testing due to its simplicity, extensibility, and rich ecosystem.


In [ ]:
import pytest
import requests
from unittest.mock import patch, MagicMock

# --- Define a sample Agent Tool ---
# This tool fetches the current stock price for a given ticker symbol.
# In a real agent, this would be exposed via a Tool decorator or similar mechanism.

class StockPriceFetcher:
    """A tool to fetch the current stock price for a given ticker symbol."""
    BASE_URL = "https://api.example.com/stock"
    API_KEY = "YOUR_MOCKED_API_KEY" # In a real app, this would be from env vars

    def __init__(self, api_key: str = None):
        self.api_key = api_key if api_key else self.API_KEY

    def fetch_price(self, ticker_symbol: str) -> dict:
        """
        Fetches the current stock price for the given ticker symbol.
        
        Args:
            ticker_symbol (str): The stock ticker symbol (e.g., "AAPL", "GOOG").
            
        Returns:
            dict: A dictionary containing the ticker, price, and timestamp,
                  or an error message if the fetch fails.
        """
        if not isinstance(ticker_symbol, str) or not ticker_symbol.strip():
            return {"error": "Invalid ticker symbol provided."}

        try:
            # Simulate an external API call
            response = requests.get(
                f"{self.BASE_URL}/{ticker_symbol}",
                params={"apiKey": self.api_key}
            )
            response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)
            data = response.json()
            
            # Simulate parsing a real API response
            if data and "price" in data and "symbol" in data:
                return {
                    "ticker": data["symbol"],
                    "price": float(data["price"]),
                    "timestamp": data.get("timestamp", "N/A")
                }
            else:
                return {"error": f"Could not parse data for {ticker_symbol}. Response: {data}"}

        except requests.exceptions.HTTPError as e:
            status_code = e.response.status_code if e.response else "Unknown"
            if status_code == 404:
                return {"error": f"Stock ticker '{ticker_symbol}' not found."}
            elif status_code == 401:
                return {"error": "Authentication failed. Invalid API Key."}
            else:
                return {"error": f"HTTP error fetching price for {ticker_symbol}: {e}"}
        except requests.exceptions.ConnectionError:
            return {"error": "Network connection error. Please check your internet."}
        except requests.exceptions.Timeout:
            return {"error": "Request timed out while fetching stock price."}
        except Exception as e:
            return {"error": f"An unexpected error occurred: {e}"}


# --- Unit Tests for StockPriceFetcher Tool ---

# We'll use pytest fixtures for setup if needed, but for simple tools,
# direct instantiation in tests is often sufficient.

@pytest.fixture
def stock_fetcher():
    """Fixture to provide a StockPriceFetcher instance."""
    return StockPriceFetcher(api_key="TEST_API_KEY")


def test_fetch_price_success(stock_fetcher):
    """Test successful fetching of a stock price."""
    # Mock the requests.get call to prevent actual network requests
    with patch('requests.get') as mock_get:
        # Configure the mock to return a successful response
        mock_response = MagicMock()
        mock_response.status_code = 200
        mock_response.json.return_value = {
            "symbol": "AAPL",
            "price": 175.50,
            "timestamp": "2026-01-01T10:00:00Z"
        }
        mock_response.raise_for_status.return_value = None # No HTTPError
        mock_get.return_value = mock_response

        result = stock_fetcher.fetch_price("AAPL")

        # Assertions
        assert result == {
            "ticker": "AAPL",
            "price": 175.50,
            "timestamp": "2026-01-01T10:00:00Z"
        }
        mock_get.assert_called_once_with(
            "https://api.example.com/stock/AAPL",
            params={"apiKey": "TEST_API_KEY"}
        )

def test_fetch_price_ticker_not_found(stock_fetcher):
    """Test handling of a 404 Not Found error from the API."""
    with patch('requests.get') as mock_get:
        mock_response = MagicMock()
        mock_response.status_code = 404
        mock_response.raise_for_status.side_effect = requests.exceptions.HTTPError("Not Found", response=mock_response)
        mock_get.return_value = mock_response

        result = stock_fetcher.fetch_price("UNKNOWN")

        assert result == {"error": "Stock ticker 'UNKNOWN' not found."}
        mock_get.assert_called_once()

def test_fetch_price_invalid_api_key(stock_fetcher):
    """Test handling of a 401 Unauthorized error from the API."""
    with patch('requests.get') as mock_get:
        mock_response = MagicMock()
        mock_response.status_code = 401
        mock_response.raise_for_status.side_effect = requests.exceptions.HTTPError("Unauthorized", response=mock_response)
        mock_get.return_value = mock_response

        result = stock_fetcher.fetch_price("MSFT")

        assert result == {"error": "Authentication failed. Invalid API Key."}
        mock_get.assert_called_once()

def test_fetch_price_network_error(stock_fetcher):
    """Test handling of a network connection error."""
    with patch('requests.get', side_effect=requests.exceptions.ConnectionError) as mock_get:
        result = stock_fetcher.fetch_price("GOOG")

        assert result == {"error": "Network connection error. Please check your internet."}
        mock_get.assert_called_once()

def test_fetch_price_timeout_error(stock_fetcher):
    """Test handling of a request timeout error."""
    with patch('requests.get', side_effect=requests.exceptions.Timeout) as mock_get:
        result = stock_fetcher.fetch_price("AMZN")

        assert result == {"error": "Request timed out while fetching stock price."}
        mock_get.assert_called_once()

def test_fetch_price_invalid_input_type(stock_fetcher):
    """Test handling of non-string ticker symbol input."""
    result = stock_fetcher.fetch_price(123)
    assert result == {"error": "Invalid ticker symbol provided."}

def test_fetch_price_empty_input(stock_fetcher):
    """Test handling of an empty ticker symbol string."""
    result = stock_fetcher.fetch_price(" ")
    assert result == {"error": "Invalid ticker symbol provided."}

def test_fetch_price_malformed_json_response(stock_fetcher):
    """Test handling of a malformed JSON response from the API."""
    with patch('requests.get') as mock_get:
        mock_response = MagicMock()
        mock_response.status_code = 200
        mock_response.json.return_value = {"status": "success", "data": "not_a_price"} # Missing 'price' key
        mock_response.raise_for_status.return_value = None
        mock_get.return_value = mock_response

        result = stock_fetcher.fetch_price("TSLA")
        assert "Could not parse data" in result["error"]
        assert "TSLA" in result["error"]
        mock_get.assert_called_once()


# To run these tests in a Jupyter environment, you would typically save them
# to a file (e.g., `test_stock_tool.py`) and run `pytest` from your terminal.
# For demonstration, we can simulate a pytest run by collecting and executing tests.

# This block is for demonstration purposes only to show how tests would run.
# In a real setup, you'd run `pytest` from your terminal.

# import sys
# from io import StringIO

# class PytestRunner:
#     def __init__(self):
#         self.output = StringIO()
#         self._stdout = sys.stdout

#     def __enter__(self):
#         sys.stdout = self.output
#         return self

#     def __exit__(self, exc_type, exc_val, exc_tb):
#         sys.stdout = self._stdout

#     def run_tests(self, module_name):
#         # This is a simplified way to run pytest programmatically.
#         # For more robust programmatic execution, use pytest.main()
#         # and capture its output.
#         # For this example, we'll just print a success message if no exceptions.
#         print(f"\n--- Simulating pytest run for {module_name} ---")
#         try:
#             # Collect all functions starting with 'test_' in the current scope
#             test_functions = [globals()[name] for name in globals() if name.startswith('test_') and callable(globals()[name])]
#             for test_func in test_functions:
#                 test_func(stock_fetcher())
#                 print(f"PASSED: {test_func.__name__}")
#             print("\nAll tests passed!")
#         except Exception as e:
#             print(f"\nFAILED: {e}")
#         print("----------------------------------------")

# # To run the simulated tests:
# # with PytestRunner() as runner:
# #     runner.run_tests(__name__)

# Since we cannot directly run `pytest` in a standard Jupyter cell and capture its output
# in a way that mimics the terminal, we'll rely on the user understanding that these
# functions are designed to be run by `pytest`.
# If you save this code as `test_stock_fetcher.py` and have `pytest` installed,
# you can run `pytest test_stock_fetcher.py` in your terminal to see the results.


### Interpreting the Code and Output

The code block above defines a `StockPriceFetcher` tool and a suite of unit tests for it using `pytest` and `unittest.mock`.

**The `StockPriceFetcher` Tool:**
*   It simulates an interaction with an external stock API. Notice the `requests.get` call and the error handling for various HTTP statuses and network issues. This is the core logic we want to test in isolation.

**The Unit Tests:**
*   Each function starting with `test_` is a separate test case. `pytest` automatically discovers and runs these.
*   **`@pytest.fixture`**: The `stock_fetcher` fixture provides a fresh instance of our tool for each test, ensuring test isolation.
*   **`patch('requests.get')`**: This is the crucial part for mocking. We use `unittest.mock.patch` to replace the actual `requests.get` function with a `MagicMock` object. This means when `stock_fetcher.fetch_price` calls `requests.get`, it's calling our mock, not making a real network request.
*   **Configuring the Mock**: We configure `mock_get` (our mocked `requests.get`) to return a `MagicMock` object that simulates a `requests.Response` object. We set its `status_code`, `json.return_value`, and `raise_for_status.side_effect` (or `return_value`) to mimic different API responses (success, 404, 401, network errors, timeouts).
*   **Assertions (`assert`)**: After calling the tool's method with mocked dependencies, we use `assert` statements to verify that the tool returned the expected output and handled errors correctly. We also use `mock_get.assert_called_once_with(...)` to ensure the tool attempted to call the external API with the correct parameters.

**Expected `pytest` Output (if run in terminal):**
When you run `pytest test_stock_fetcher.py` (assuming you save the tests in that file), you would see output similar to this:

```
============================= test session starts ==============================
platform linux -- Python 3.10.12, pytest-7.4.0, pluggy-1.3.0
rootdir: /path/to/your/project
collected 8 items

test_stock_fetcher.py ...........                                        [100%]

============================== 8 passed in 0.05s ===============================
```

Each dot `.` represents a passed test. If a test fails, you'd see an `F` and a detailed traceback indicating where the assertion failed.

### Performance Trade-offs and Use Cases

**Performance:**
Unit tests are designed to be fast. By mocking external dependencies, we eliminate slow network calls, database queries, or complex computations. This allows developers to run hundreds or thousands of unit tests in seconds, providing immediate feedback during development.

**Typical Use Cases for Unit Testing Agent Tools:**
1.  **Core Logic Validation**: Ensure the tool's internal algorithms, data transformations, and business rules are correct.
2.  **Input Validation**: Verify that the tool correctly handles valid, invalid, and edge-case inputs (e.g., empty strings, wrong data types, out-of-range values).
3.  **Error Handling**: Confirm that the tool gracefully handles various error conditions from external services (e.g., API rate limits, authentication failures, network outages, malformed responses).
4.  **Security**: Test for potential vulnerabilities like injection attacks if the tool processes user-controlled input that interacts with a shell or database.
5.  **Regression Prevention**: Catch unintended side effects or bugs introduced when refactoring or adding new features to a tool.
6.  **Documentation**: Well-written unit tests serve as executable documentation for how a tool is expected to behave.

**Distinction from Integration/End-to-End Tests:**
While unit tests are crucial, they don't replace integration or end-to-end tests. Unit tests verify the tool in isolation. Integration tests would verify that the agent can *correctly choose and invoke* the tool, and that the tool's output is correctly processed by the agent. End-to-end tests would cover the entire agentic workflow, including the LLM's reasoning, tool selection, execution, and final output, often involving real external systems. Each testing level serves a different purpose, building confidence in the overall agent system.


### Resources

*   **`pytest` Documentation**: The official documentation for `pytest`, a powerful and easy-to-use Python testing framework. [https://docs.pytest.org/en/stable/](https://docs.pytest.org/en/stable/)
*   **`unittest.mock` Documentation**: Learn more about Python's built-in mocking library, essential for isolating code during testing. [https://docs.python.org/3/library/unittest.mock.html](https://docs.python.org/3/library/unittest.mock.html)
*   **`requests` Library**: The HTTP library for Python, commonly used in agent tools for API interactions. [https://requests.readthedocs.io/en/latest/](https://requests.readthedocs.io/en/latest/)
*   **Pytest-Mock Plugin**: A `pytest` plugin that simplifies mocking with `unittest.mock` fixtures. While we used direct `patch` here, `pytest-mock` offers a more `pytest`-native way to mock. [https://pytest-mock.readthedocs.io/en/latest/](https://pytest-mock.readthedocs.io/en/latest/)
*   **Testing Strategies for AI Applications**: While not specific to unit testing tools, this provides broader context on testing AI systems. (Search for recent articles on "testing AI agents" or "MLOps testing strategies" from sources like Google AI, Microsoft AI, or leading MLOps platforms for 2026 insights).
